# E-Commerce Data Analysis - Functions Library

  
> **Purpose:** Reusable functions for data cleaning, validation, and visualization

---

## Table of Contents

| # | Section | Key Functions |
|---|---------|---------------|
| 1 | Data Loading | `read_csv_dedup`, `read_and_merge_vendor_files` |
| 2 | Data Profiling | `profile_dashboard` |
| 3 | Data Quality | `drop_rows_missing_pk`, `apply_upper_lower_cols` |
| 4 | Datetime | `normalize_datetime_string`, `summarize_date_formats` |
| 5 | Type Casting | `typecast_df` |
| 6 | Filtering | `filter_df_by_datetime`, `fill_from_mapping` |
| 7 | EDA & Outliers | `plot_histogram`, `plot_frequency`, `analyze_numeric_outliers` |
| 8 | Column Selection | `keep_columns_in_df` |
| 9 | Merging | `merge_many_df`, `revenue_analysis` |
| 10 | Timezone | `to_company_time` |
| 11 | Missing Values | `fill_categorical_na`, `fill_string_na` |

---

**How to use:** Run this notebook first, then in your analysis notebook:
```python
%run functions.ipynb
```


## Imports

These libraries handle data manipulation, visualization, and timezone conversions.

In [4]:
from __future__ import annotations

import sys
import re
import pandas as pd
import numpy as np
from pathlib import Path
from typing import Any, Dict, List, Optional, Sequence, Tuple, Union

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display

# Date and time utilities
from pandas.tseries.offsets import YearEnd, MonthEnd, WeekOfMonth
from matplotlib.lines import Line2D
from zoneinfo import ZoneInfo

In [5]:
# PLOTTING STYLE CONFIGURATION

# These settings ensure consistent, professional-looking charts

# Color palette for visualizations
COLORS = {
    'primary': '#4C72B0',      # Steel blue - main bars/lines
    'secondary': '#55A868',    # Green - secondary elements
    'accent': '#C44E52',       # Red - highlights/warnings
    'neutral': '#8C8C8C',      # Gray - backgrounds/grids
    'success': '#27AE60',      # Green - positive indicators
    'warning': '#F39C12',      # Orange - caution
    'danger': '#E74C3C',       # Red - negative/outliers
}

# Seaborn style
sns.set_style("whitegrid")
sns.set_palette("husl")

# Matplotlib defaults
plt.rcParams.update({
    'figure.figsize': [12, 6],
    'figure.dpi': 100,
    'figure.facecolor': 'white',
    'axes.facecolor': '#FAFAFA',
    'axes.titlesize': 14,
    'axes.titleweight': 'bold',
    'axes.labelsize': 12,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.size': 11,
    'legend.fontsize': 10,
    'legend.framealpha': 0.9,
})

# Plotly template
import plotly.io as pio
pio.templates.default = "plotly_white"

print(" Visualization styles configured")

 Visualization styles configured


### Visualization Configuration

The cell above sets up:
- **Color palette** - Consistent colors across all charts
- **Seaborn style** - Clean whitegrid background
- **Matplotlib defaults** - Professional fonts, sizes, and layout
- **Plotly template** - Matching white theme for interactive plots


---
## 1. Data Loading Functions

These functions read CSV files and handle common data quality issues at the source:
- Removing duplicate rows that could skew analysis
- Cleaning up whitespace in text columns
- Standardizing column order across vendor files

In [4]:
def read_csv_dedup(path):
    """
    Read a CSV file and clean it up for analysis.
    
    Steps:
    1. Load the CSV file
    2. Strip whitespace from text columns ("ABC " becomes "ABC")
    3. Convert empty strings and null-like values to proper NaN
    4. Remove exact duplicate rows (prints count before removing)
    
    Parameters
    ----------
    path : str or Path
        File path to the CSV file
    
    Returns
    -------
    DataFrame
        Cleaned data ready for analysis
    """
    df = pd.read_csv(path)
    
    # Store original row count
    original_rows = len(df)
    
    # Find and clean text columns
    text_cols = df.select_dtypes(include=["object", "string"]).columns
    df[text_cols] = df[text_cols].apply(lambda col: col.astype("string").str.strip())
    
    # Standardize null representations
    df = df.replace({"": pd.NA, "nan": pd.NA, "None": pd.NA, "NULL": pd.NA})
    
    # Count duplicates BEFORE removing
    num_duplicates = df.duplicated(keep="first").sum()
    
    # Remove duplicates
    df = df.drop_duplicates(keep="first").reset_index(drop=True)
    
    # Print duplicate report
    if num_duplicates > 0:
        print(f" {path.name if hasattr(path, 'name') else path}:")
        print(f"   Rows before: {original_rows:,}")
        print(f"   Duplicates found: {num_duplicates:,} ({num_duplicates/original_rows*100:.2f}%)")
        print(f"   Rows after: {len(df):,}")
    else:
        print(f" {path.name if hasattr(path, 'name') else path}: No duplicates ({original_rows:,} rows)")
    
    return df

In [9]:
def reorganize_columns(df, column_order):
    """
    Rearrange columns to match a standard order.
    
    Useful when combining multiple files that have columns in different orders.
    Columns not in the order list are placed at the end.
    
    Parameters
    ----------
    df : DataFrame
        Data to reorganize
    column_order : list
        Desired column order
    
    Returns
    -------
    DataFrame
        Reordered columns
    """
    df = df.copy()
    
    # Columns in both the data and desired order
    matching_cols = [c for c in column_order if c in df.columns]
    
    # Extra columns not in the desired order
    extra_cols = [c for c in df.columns if c not in column_order]
    
    return df[matching_cols + extra_cols]

In [10]:
def read_and_merge_vendor_files(folder_path, column_order):
    """
    Read and combine multiple vendor CSV files into one dataset.
    
    Handles common inconsistencies across vendor files:
    - Some have 'profit_margin' as decimal (0.15), others as percent (15)
    - Column order differs between files
    - Duplicates may exist within and across files
    
    Parameters
    ----------
    folder_path : str or Path
        Path to folder containing vendor CSV files
    column_order : list
        Standard column order to use for all files
    
    Returns
    -------
    DataFrame
        Combined vendor data
    
    Raises
    ------
    ValueError
        If no CSV files found in folder
    """
    all_frames = []
    folder_path = Path(folder_path)
    csv_files = list(folder_path.glob("*.csv"))

    if not csv_files:
        raise ValueError(f"No CSV files found in {folder_path}")

    for file_path in csv_files:
        df = read_csv_dedup(file_path)

        # Normalize profit margin column
        if "profit_margin" in df.columns:
            df["profit_margin"] = pd.to_numeric(df["profit_margin"], errors="coerce")
        elif "pct_profit_margin" in df.columns:
            # Convert percent to decimal (15 -> 0.15)
            df["profit_margin"] = pd.to_numeric(df["pct_profit_margin"], errors="coerce") / 100
            df = df.drop(columns=["pct_profit_margin"])
        else:
            df["profit_margin"] = np.nan

        df = reorganize_columns(df, column_order)
        all_frames.append(df)

    return pd.concat(all_frames, ignore_index=True)

---
## 2. Data Profiling

Quick sanity-checking functions to understand dataset shape, types, and quality issues.

In [12]:
def profile_dashboard(df: pd.DataFrame, name: str = "dataset", 
                     top_n_missing: int = 14, top_n_card: int = 10):
    """
    Create a quick summary dashboard for a dataset.
    
    Shows:
    - Basic stats (rows, columns, missing data, duplicates)
    - Which columns have missing values
    - Data type distribution
    - Text column cardinality (how many unique values)
    - Numeric distributions
    
    This is useful for quickly spotting data quality issues.
    
    Parameters:
    -----------
    df : DataFrame
        Dataset to analyze
    name : str
        Name for the dataset (for display)
    top_n_missing : int
        How many columns with missing data to show
    top_n_card : int
        How many high-cardinality columns to show
    """
    # Basic summary statistics
    summary = {
        "rows": len(df),
        "cols": df.shape[1],
        "missing_cells_%": round(df.isna().mean().mean() * 100, 2),
        "dup_rows_%": round(df.duplicated().mean() * 100, 2),
    }
    display(pd.DataFrame([summary], index=[name]))

    # Find columns with missing data
    missing = (df.isna().mean() * 100).sort_values(ascending=False)
    missing = missing[missing > 0].head(top_n_missing)

    # Count data types
    dtype_groups = []
    for dtype in df.dtypes.astype(str):
        if "int" in dtype or "float" in dtype:
            dtype_groups.append("numeric")
        elif "datetime" in dtype:
            dtype_groups.append("datetime")
        elif dtype in ["bool", "boolean"]:
            dtype_groups.append("boolean")
        else:
            dtype_groups.append("text")
    dtype_counts = pd.Series(dtype_groups).value_counts()

    # Check cardinality of text columns
    text_cols = df.select_dtypes(include=["object", "string"]).columns.tolist()
    cardinality = pd.Series(
        {c: df[c].nunique(dropna=True) for c in text_cols}
    ).sort_values(ascending=False).head(top_n_card)

    numeric_cols = df.select_dtypes(include=["int64", "float64", "int32", "float32"]).columns.tolist()

    # Create 4-panel visualization
    fig, axes = plt.subplots(1, 4, figsize=(24, 6))

    # Panel 1: Missing data
    if missing.empty:
        axes[0].set_title("Missing Data\n(none found)")
        axes[0].axis("off")
    else:
        axes[0].barh(missing.index[::-1], missing.values[::-1], color="coral")
        axes[0].set_title(f"Top {len(missing)} Columns with Missing Data")
        axes[0].set_xlabel("% missing")

    # Panel 2: Data types
    axes[1].bar(dtype_counts.index, dtype_counts.values, color="skyblue")
    axes[1].set_title("Column Types")
    axes[1].set_ylabel("# of columns")
    axes[1].tick_params(axis="x", rotation=30)

    # Panel 3: Cardinality
    if cardinality.empty:
        axes[2].set_title("High Cardinality Columns\n(none)")
        axes[2].axis("off")
    else:
        axes[2].barh(cardinality.index[::-1], cardinality.values[::-1], color="lightgreen")
        axes[2].set_title("Text Columns (# unique values)")
        axes[2].set_xlabel("unique count")

    # Panel 4: Numeric distributions
    if not numeric_cols:
        axes[3].set_title("Numeric Distributions\n(no numeric columns)")
        axes[3].axis("off")
    else:
        for col in numeric_cols[:5]:  # Show up to 5 numeric columns
            axes[3].hist(df[col].dropna(), alpha=0.5, label=col, bins=30)
        axes[3].set_title("Numeric Column Distributions")
        axes[3].legend(fontsize=8)

    plt.tight_layout()
    plt.show()

---
## 3. Data Quality and Casing

Functions to reduce key-mismatch risk in joins and prevent data loss from missing primary keys.

In [14]:
def drop_rows_missing_pk(df: pd.DataFrame, pk: Union[str, List[str]]) -> pd.DataFrame:
    """
    Drop rows where the primary key is missing.
    
    Primary keys (like order_id, buyer_id) should never be null.
    This function treats empty strings and null-like values as missing.
    
    Parameters
    ----------
    df : DataFrame
        Data to clean
    pk : str or list
        Primary key column(s) - can be single column or composite key
    
    Returns
    -------
    DataFrame
        Rows with complete primary keys
    
    Notes
    -----
    Does NOT enforce uniqueness - multiple rows can share the same PK.
    """
    pk_cols = [pk] if isinstance(pk, str) else list(pk)
    out = df.copy()

    # Normalize "missing-like" strings for PK columns
    for col in pk_cols:
        if col in out.columns:
            s = out[col]
            if pd.api.types.is_object_dtype(s) or pd.api.types.is_string_dtype(s):
                out[col] = (
                    s.astype("string")
                     .str.strip()
                     .replace({"": pd.NA, "nan": pd.NA, "None": pd.NA, "NULL": pd.NA})
                )

    # Drop rows where ANY PK column is missing
    out = out.dropna(subset=pk_cols, how="any")
    return out

In [15]:
def apply_upper_lower_cols(df: pd.DataFrame, upper_cols=None, lower_cols=None) -> pd.DataFrame:
    """
    Convert specified columns to upper or lower case.
    
    Useful for standardizing text columns before joins or grouping.
    
    Parameters
    ----------
    df : DataFrame
        Data to modify
    upper_cols : list or None
        Columns to convert to UPPERCASE
    lower_cols : list or None
        Columns to convert to lowercase
    
    Returns
    -------
    DataFrame
        Data with case-converted columns
    """
    out = df.copy()
    upper_cols = upper_cols or []
    lower_cols = lower_cols or []

    for col in upper_cols:
        if col in out.columns:
            out[col] = out[col].astype("string").str.upper()

    for col in lower_cols:
        if col in out.columns:
            out[col] = out[col].astype("string").str.lower()

    return out

---
## 4. Datetime Normalization

Datetime inputs come in mixed formats. These functions normalize them for consistent filtering and time-based logic.

In [17]:
def summarize_date_formats(series: pd.Series) -> dict:
    """
    Classify datetime strings into format categories.
    
    Useful for understanding what date formats exist in a column
    before deciding how to parse them.
    
    Categories:
    - Datetime: has time component (HH:MM)
    - Date Only: has date separators but no time
    - Epoch: unix timestamp (9+ digits)
    - Missing/NA: null or unparseable
    
    Parameters
    ----------
    series : Series
        Column containing date/time strings
    
    Returns
    -------
    dict
        Counts for each format category
    """
    if not isinstance(series, pd.Series):
        raise TypeError("summarize_date_formats expects a pandas Series.")

    s = series.astype("string").str.strip()
    s = s.replace({"": pd.NA, "nan": pd.NA, "None": pd.NA, "NULL": pd.NA})

    missing = s.isna().sum()

    # Epoch: 9+ digits, optional decimal
    epoch = s.str.fullmatch(r"\d{9,}(?:\.\d+)?", na=False)

    # Time component present
    has_time = s.str.contains(r"\b\d{1,2}:\d{2}\b", regex=True, na=False)

    # Date separators present
    has_date_sep = s.str.contains(r"[-/]", regex=True, na=False)

    counts = {
        "Datetime": int((~epoch & has_time).sum()),
        "Date Only": int((~epoch & ~has_time & has_date_sep).sum()),
        "Epoch": int(epoch.sum()),
        "Missing/NA": int(missing + (~epoch & ~has_time & ~has_date_sep & s.notna()).sum()),
    }
    return counts

In [18]:
def normalize_datetime_string(s: pd.Series) -> pd.Series:
    """
    Normalize mixed-format datetime strings to a standard format.
    
    Output formats:
    - With time: 'MM/DD/YYYY HH:MM'
    - Date only: 'MM/DD/YYYY'
    
    Supported input formats:
    - Epoch seconds (10 digits)
    - M/D/YY [HH:MM]
    - DD-Mon-YY [HH:MM]
    
    Parameters
    ----------
    s : Series
        Column with mixed datetime formats
    
    Returns
    -------
    Series
        Normalized datetime strings (or NA for unparseable values)
    """
    if not isinstance(s, pd.Series):
        raise TypeError("normalize_datetime_string expects a pandas Series.")

    raw = s.astype("string").str.strip()
    raw = raw.replace({"": pd.NA, "nan": pd.NA, "None": pd.NA, "NULL": pd.NA})

    out_dt = pd.Series(pd.NaT, index=raw.index)
    has_time = pd.Series(False, index=raw.index)

    time_token = raw.str.contains(r"\b\d{1,2}:\d{2}\b", regex=True, na=False)

    # Handle epoch seconds (10 digits)
    m = raw.str.fullmatch(r"\d{10}", na=False)
    if m.any():
        out_dt.loc[m] = (
            pd.to_datetime(raw.loc[m].astype("int64"), unit="s", utc=True, errors="coerce")
              .dt.tz_convert(None)
        )
        has_time.loc[m] = True

    def _parse(mask: pd.Series, fmt_time: str, fmt_date: str):
        """Helper to parse date-only or date+time for a given mask and formats."""
        if not mask.any():
            return
        dt_time = pd.to_datetime(raw.loc[mask], format=fmt_time, errors="coerce")
        dt_date = pd.to_datetime(raw.loc[mask], format=fmt_date, errors="coerce")
        out_dt.loc[mask] = dt_time.fillna(dt_date)
        has_time.loc[mask] = time_token.loc[mask] & out_dt.loc[mask].notna()

    # Handle M/D/YY [HH:MM]
    m = out_dt.isna() & raw.str.fullmatch(r"\d{1,2}/\d{1,2}/\d{2}(?:\s+\d{1,2}:\d{2})?", na=False)
    _parse(m, "%m/%d/%y %H:%M", "%m/%d/%y")

    # Handle DD-Mon-YY [HH:MM]
    m = out_dt.isna() & raw.str.fullmatch(r"\d{1,2}-[A-Za-z]{3}-\d{2}(?:\s+\d{1,2}:\d{2})?", na=False)
    _parse(m, "%d-%b-%y %H:%M", "%d-%b-%y")

    # Build output strings
    out = pd.Series(pd.NA, index=raw.index, dtype="string")
    ok = out_dt.notna()

    out.loc[ok & has_time] = out_dt.loc[ok & has_time].dt.strftime("%m/%d/%Y %H:%M")
    out.loc[ok & ~has_time] = out_dt.loc[ok & ~has_time].dt.strftime("%m/%d/%Y")

    return out

---
## 5. Type Casting

Ensure correct data types for numeric operations and categorical grouping.

In [20]:
def typecast_df(df: pd.DataFrame, cols_meta: dict) -> pd.DataFrame:
    """
    Convert columns to correct data types based on a metadata dictionary.
    
    Parameters
    ----------
    df : DataFrame
        Data to convert
    cols_meta : dict
        Dictionary specifying column types:
        - 'string_cols': text columns
        - 'num_cols': numeric columns (handles $, %, commas)
        - 'bool_cols': boolean columns (handles various truthy/falsy values)
        - 'datetime_cols': datetime columns (expects normalized format)
        - 'category_cols': categorical columns
    
    Returns
    -------
    DataFrame
        Data with corrected types
    
    Example
    -------
    >>> metadata = {
    ...     "num_cols": ["price", "quantity"],
    ...     "datetime_cols": ["order_date"],
    ...     "category_cols": ["customer_segment"]
    ... }
    >>> df = typecast_df(df, metadata)
    """
    if not isinstance(cols_meta, dict):
        raise TypeError("cols_meta must be a dict with keys like 'string_cols', 'num_cols', etc.")

    out = df.copy()

    string_cols = cols_meta.get("string_cols", []) or []
    num_cols = cols_meta.get("num_cols", []) or []
    bool_cols = cols_meta.get("bool_cols", []) or []
    datetime_cols = cols_meta.get("datetime_cols", []) or []
    category_cols = cols_meta.get("category_cols", []) or []

    # Datetime columns (expects pre-normalized format from normalize_datetime_string)
    for col in datetime_cols:
        if col in out.columns:
            s = out[col].astype("string")
            dt_time = pd.to_datetime(s, format="%m/%d/%Y %H:%M", errors="coerce")
            dt_date = pd.to_datetime(s, format="%m/%d/%Y", errors="coerce")
            out[col] = dt_time.fillna(dt_date)

    # String columns
    for col in string_cols:
        if col in out.columns:
            out[col] = out[col].astype("string")

    # Numeric columns (remove currency symbols, percentages, commas)
    for col in num_cols:
        if col in out.columns:
            s = out[col].astype("string")
            s = (
                s.str.replace("$", "", regex=False)
                 .str.replace("%", "", regex=False)
                 .str.replace(",", "", regex=False)
                 .str.strip()
            )
            out[col] = pd.to_numeric(s, errors="coerce")

    # Boolean columns
    bool_map = {
        "1": True, "true": True, "t": True, "yes": True, "y": True,
        "0": False, "false": False, "f": False, "no": False, "n": False,
        "2": False
    }
    for col in bool_cols:
        if col in out.columns:
            s = out[col].astype("string").str.strip().str.lower()
            s = s.replace({"": pd.NA})
            out[col] = s.map(bool_map).fillna(False).astype(bool)

    # Category columns
    for col in category_cols:
        if col in out.columns:
            out[col] = out[col].astype("category")

    return out

---
## 6. Filtering and Imputation

Filter data to analysis windows and fill missing values using stable key mappings.

In [22]:
def _parse_flexible_bound(x: str, is_end: bool) -> pd.Timestamp:
    """
    Parse flexible datetime inputs into a Timestamp.
    
    If is_end=True, returns the END of the specified period:
    - "2023" -> end of year
    - "2023-11" -> end of month
    - "2023-11-24" -> end of day (23:59)
    - "2023-11-24 14" -> end of hour (:59)
    
    Parameters
    ----------
    x : str
        Date/time string in various formats
    is_end : bool
        If True, return end of period; if False, return start
    
    Returns
    -------
    Timestamp
        Parsed timestamp
    """
    ts = pd.to_datetime(str(x), errors='raise')
    
    if not is_end:
        return ts
    
    x_str = str(x).strip().replace("/", "-")
    
    # Year only
    if re.fullmatch(r"\d{4}$", x_str):
        return ts + YearEnd(0)
    
    # Year-Month
    if re.fullmatch(r"\d{4}-\d{1,2}$", x_str):
        return ts + MonthEnd(0)
    
    # Date only
    if re.fullmatch(r"\d{4}-\d{1,2}-\d{1,2}$", x_str):
        return ts.replace(hour=23, minute=59)
    
    # Date with hour
    if re.fullmatch(r"\d{4}-\d{1,2}-\d{1,2}\s+\d{1,2}$", x_str):
        return ts.replace(minute=59)
    
    return ts

In [23]:
def filter_df_by_datetime(df: pd.DataFrame, date_col: str, start: str, end: str):
    """
    Filter DataFrame to a date range with flexible inputs.
    
    Accepts various date formats for start/end:
    - YYYY
    - YYYY-MM
    - YYYY-MM-DD
    - YYYY-MM-DD HH
    - YYYY-MM-DD HH:MM
    
    The end bound is automatically inclusive to end-of-period.
    Rows with null dates are preserved.
    
    Parameters
    ----------
    df : DataFrame
        Data to filter
    date_col : str
        Name of datetime column
    start : str
        Start of date range
    end : str
        End of date range
    
    Returns
    -------
    DataFrame
        Filtered data
    """
    out = df.copy()
    if date_col not in out.columns:
        raise KeyError(f"date_col not found: {date_col}")
        
    before = len(out)
    parsed_col = f"{date_col}_dt"

    out[parsed_col] = pd.to_datetime(out[date_col], errors="coerce")

    start_dt = _parse_flexible_bound(start, is_end=False)
    end_dt = _parse_flexible_bound(end, is_end=True)

    mask = out[parsed_col].isna() | out[parsed_col].between(start_dt, end_dt, inclusive="both")
    out = out.loc[mask].copy()
    after = len(out)

    print(f"datetime filter '{date_col}': start={start_dt} end={end_dt}")
    print(f"rows: before={before:,} after={after:,} dropped={before-after:,}")

    return out.drop(columns=[parsed_col])

In [24]:
def fill_from_mapping(key: pd.Series, target: pd.Series) -> pd.Series:
    """
    Fill missing values in target using the most common value for each key.
    
    For example, if buyer_id "ABC" usually has timezone "America/New_York",
    fill missing timezones for that buyer with "America/New_York".
    
    Parameters
    ----------
    key : Series
        Key column (e.g., buyer_id)
    target : Series
        Column with missing values to fill (e.g., timezone)
    
    Returns
    -------
    Series
        Target with missing values filled where possible
    """
    tmp = pd.DataFrame({"k": key, "t": target}).dropna(subset=["k", "t"])

    # Count occurrences of each key-target pair
    counts = tmp.groupby(["k", "t"], sort=False, observed=True).size()

    # For each key, get the most common target value
    mapping = counts.groupby(level=0, sort=False, observed=True).idxmax().map(lambda kt: kt[1])

    # Fill missing values
    out = target.copy()
    mask = out.isna() & key.notna()
    out.loc[mask] = key.loc[mask].map(mapping)
    return out

---
## 7. EDA Plotting and Outlier Analysis

Functions to identify unrealistic values that could distort aggregates and metrics.

In [26]:
def plot_histogram(series: pd.Series, title: str = None, bins: int = 30, 
                   color: str = "#4C72B0", show_stats: bool = True):
    """
    Create a polished histogram for a numeric column with statistical annotations.
    
    Features:
    - Clean, modern styling with customizable colors
    - Statistical summary (mean, median, std) displayed on chart
    - Vertical lines showing mean and median
    - Formatted axis labels with thousand separators
    
    Parameters
    ----------
    series : Series
        Numeric data to plot
    title : str, optional
        Chart title (defaults to "Distribution of {column_name}")
    bins : int, default 30
        Number of bins in histogram
    color : str, default "#4C72B0"
        Bar color (hex code or named color)
    show_stats : bool, default True
        Whether to display statistical summary box
    
    Returns
    -------
    tuple
        (fig, ax) matplotlib figure and axis objects for further customization
    
    Example
    -------
    >>> fig, ax = plot_histogram(df['price'], title='Price Distribution', color='steelblue')
    """
    # Prepare data
    data = series.dropna()
    col_name = series.name or "Value"
    
    if title is None:
        title = f"Distribution of {col_name}"
    
    # Create figure with clean style
    fig, ax = plt.subplots(figsize=(11, 6), facecolor='white')
    ax.set_facecolor('#FAFAFA')
    
    # Plot histogram
    n, bin_edges, patches = ax.hist(
        data, bins=bins, 
        color=color, 
        edgecolor='white', 
        alpha=0.85,
        linewidth=1.2
    )
    
    # Calculate statistics
    mean_val = data.mean()
    median_val = data.median()
    std_val = data.std()
    
    # Add mean and median lines
    ax.axvline(mean_val, color='#E74C3C', linestyle='--', linewidth=2, label=f'Mean: {mean_val:,.2f}')
    ax.axvline(median_val, color='#27AE60', linestyle='-.', linewidth=2, label=f'Median: {median_val:,.2f}')
    
    # Style the plot
    ax.set_xlabel(col_name, fontsize=12, fontweight='medium', labelpad=10)
    ax.set_ylabel("Frequency", fontsize=12, fontweight='medium', labelpad=10)
    ax.set_title(title, fontsize=14, fontweight='bold', pad=15)
    
    # Format axis with thousand separators
    ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'{x:,.0f}'))
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'{x:,.0f}'))
    
    # Add subtle grid
    ax.grid(axis='y', alpha=0.3, linestyle='-', linewidth=0.5)
    ax.set_axisbelow(True)
    
    # Remove top and right spines
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_color('#CCCCCC')
    ax.spines['bottom'].set_color('#CCCCCC')
    
    # Add statistics box if requested
    if show_stats:
        stats_text = f"n = {len(data):,}\nMean = {mean_val:,.2f}\nMedian = {median_val:,.2f}\nStd = {std_val:,.2f}"
        props = dict(boxstyle='round,pad=0.5', facecolor='white', edgecolor='#CCCCCC', alpha=0.9)
        ax.text(0.97, 0.97, stats_text, transform=ax.transAxes, fontsize=10,
                verticalalignment='top', horizontalalignment='right', bbox=props,
                fontfamily='monospace')
    
    # Add legend
    ax.legend(loc='upper right', framealpha=0.9, fontsize=10, 
              bbox_to_anchor=(0.99, 0.75))
    
    plt.tight_layout()
    plt.show()
    
    return fig, ax

In [27]:
def plot_frequency(series: pd.Series, title: str = None, top_n: int = 20,
                   color: str = "#5DADE2", show_pct: bool = True):
    """
    Create a polished horizontal bar chart showing value frequencies.
    
    Features:
    - Clean, modern styling with gradient-like color intensity
    - Percentage labels alongside counts
    - Value labels on bars for easy reading
    - Smart text positioning (inside/outside bars based on size)
    
    Parameters
    ----------
    series : Series
        Categorical or discrete numeric data to plot
    title : str, optional
        Chart title (defaults to "Top {n} Values in {column_name}")
    top_n : int, default 20
        Number of categories to show
    color : str, default "#5DADE2"
        Base bar color (hex code or named color)
    show_pct : bool, default True
        Whether to show percentage alongside count
    
    Returns
    -------
    tuple
        (value_counts, fig, ax) - full value counts Series and matplotlib objects
    
    Example
    -------
    >>> counts, fig, ax = plot_frequency(df['category'], top_n=15, color='teal')
    """
    # Prepare data
    data = series.dropna()
    col_name = series.name or "Value"
    full_counts = data.value_counts()
    counts = full_counts.head(top_n)
    total = len(data)
    
    if title is None:
        if len(full_counts) > top_n:
            title = f"Top {top_n} of {len(full_counts)} Unique Values in {col_name}"
        else:
            title = f"Value Distribution: {col_name}"
    
    # Create figure
    fig_height = max(6, min(12, len(counts) * 0.4 + 2))
    fig, ax = plt.subplots(figsize=(12, fig_height), facecolor='white')
    ax.set_facecolor('#FAFAFA')
    
    # Create color gradient (darker for higher values)
    max_count = counts.max()
    colors = [plt.cm.Blues(0.4 + 0.5 * (c / max_count)) for c in counts.values]
    
    # Plot horizontal bars
    y_pos = range(len(counts))
    bars = ax.barh(y_pos, counts.values, color=colors, edgecolor='white', linewidth=0.8, height=0.7)
    
    # Add value labels on bars
    for i, (bar, count) in enumerate(zip(bars, counts.values)):
        pct = count / total * 100
        if show_pct:
            label = f"{count:,} ({pct:.1f}%)"
        else:
            label = f"{count:,}"
        
        # Position text inside or outside bar based on bar width
        if bar.get_width() > max_count * 0.3:
            ax.text(bar.get_width() - max_count * 0.02, bar.get_y() + bar.get_height()/2,
                   label, va='center', ha='right', fontsize=10, color='white', fontweight='medium')
        else:
            ax.text(bar.get_width() + max_count * 0.01, bar.get_y() + bar.get_height()/2,
                   label, va='center', ha='left', fontsize=10, color='#333333')
    
    # Style the plot
    ax.set_yticks(y_pos)
    ax.set_yticklabels(counts.index, fontsize=11)
    ax.set_xlabel("Count", fontsize=12, fontweight='medium', labelpad=10)
    ax.set_title(title, fontsize=14, fontweight='bold', pad=15)
    
    # Invert y-axis so highest is at top
    ax.invert_yaxis()
    
    # Format x-axis with thousand separators
    ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'{x:,.0f}'))
    
    # Add subtle grid
    ax.grid(axis='x', alpha=0.3, linestyle='-', linewidth=0.5)
    ax.set_axisbelow(True)
    
    # Remove spines
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_visible(False)
    ax.spines['bottom'].set_color('#CCCCCC')
    
    # Add summary annotation
    summary = f"Showing {len(counts):,} of {len(full_counts):,} unique values | Total: {total:,} records"
    ax.text(0.5, -0.08, summary, transform=ax.transAxes, fontsize=10,
            ha='center', color='#666666', style='italic')
    
    plt.tight_layout()
    plt.subplots_adjust(bottom=0.12)
    plt.show()
    
    return full_counts, fig, ax

In [28]:
def analyze_numeric_outliers(series: pd.Series) -> pd.DataFrame:
    """
    Audit outliers in a numeric column using common cut rules.
    
    Methods tested:
    1. IQR (1.5x) - standard box plot whiskers
    2. 99.9% percentile (0.1%–99.9%)
    3. 99.99% percentile (0.01%–99.99%)
    
    Parameters
    ----------
    series : Series
        Numeric column to analyze
    
    Returns
    -------
    DataFrame
        Summary with outlier count, bounds, and post-filter range for each method
    """
    if not isinstance(series, pd.Series):
        raise TypeError("analyze_numeric_outliers expects a pandas Series.")
    if not pd.api.types.is_numeric_dtype(series):
        raise TypeError(f"Numeric Series required. Got: {series.dtype}")

    x = pd.to_numeric(series, errors="coerce").dropna()
    if x.empty:
        print(f"[outliers] '{series.name or 'series'}' has no numeric values.")
        return pd.DataFrame(columns=["method", "outlier_count", "min_post", "max_post", "lower", "upper"])

    def audit(method: str, lower: float, upper: float) -> dict:
        keep = x[(x >= lower) & (x <= upper)]
        return {
            "method": method,
            "outlier_count": int(len(x) - len(keep)),
            "min_post": float(keep.min()) if not keep.empty else np.nan,
            "max_post": float(keep.max()) if not keep.empty else np.nan,
            "lower": float(lower),
            "upper": float(upper),
        }

    results = []

    # IQR method
    q1, q3 = x.quantile([0.25, 0.75])
    iqr = q3 - q1
    results.append(audit("IQR (1.5x)", q1 - 1.5 * iqr, q3 + 1.5 * iqr))

    # 99.9% percentile
    lo, hi = x.quantile([0.001, 0.999])
    results.append(audit("Percentile (0.1–99.9)", lo, hi))

    # 99.99% percentile
    lo, hi = x.quantile([0.0001, 0.9999])
    results.append(audit("Percentile (0.01–99.99)", lo, hi))

    out = pd.DataFrame(results)

    print(f"Outlier Audit: {series.name or 'series'}")
    print(out.to_string(index=False))
    print("-" * 75)

    return out

In [29]:
def trim_outliers_by_percentile(df, column_name, p=0.999, keep_na=True):
    """
    Remove rows outside the [1-p, p] percentile range.
    
    Useful for removing extreme outliers while preserving NaN values.
    
    Parameters
    ----------
    df : DataFrame
        Data to trim
    column_name : str
        Numeric column to check for outliers
    p : float
        Upper percentile (default 0.999 = keep middle 99.8%)
    keep_na : bool
        If True, preserve rows with missing values
    
    Returns
    -------
    DataFrame
        Data with outliers removed
    """
    # Calculate boundaries
    lower_percentile = 1 - p
    lower_bound = df[column_name].quantile(lower_percentile, interpolation='nearest')
    upper_bound = df[column_name].quantile(p, interpolation='nearest')

    # Build masks
    within_bounds = (df[column_name] >= lower_bound) & (df[column_name] <= upper_bound)
    is_na = df[column_name].isna()

    # Apply filter
    if keep_na:
        clean_df = df[within_bounds | is_na].copy()
    else:
        clean_df = df[within_bounds].copy()

    # Report
    rows_removed = len(df) - len(clean_df)
    print(f"--- Trimming Report: {column_name} ---")
    print(f"Kept Range:  [{lower_bound}, {upper_bound}]")
    print(f"Keep NaNs:   {keep_na}")
    print(f"Dropped:     {rows_removed} rows")
    print(f"Retention:   {(len(clean_df)/len(df))*100:.2f}%")
    print("-" * 35)
    
    return clean_df

---
## 8. Column Selection

In [31]:
def keep_columns_in_df(df, cols_to_keep):
    """
    Keep only specified columns that exist in the DataFrame.
    
    Safely handles cases where some requested columns don't exist.
    
    Parameters
    ----------
    df : DataFrame
        Data to select from
    cols_to_keep : list
        Columns to keep
    
    Returns
    -------
    DataFrame
        Subset of columns
    """
    existing_cols = [c for c in cols_to_keep if c in df.columns]
    return df[existing_cols].copy()

---
## 9. Data Merging and Gap Analysis

Multi-merge utilities and missingness diagnostics to validate that joins work correctly.

In [33]:
def merge_many_df(base: pd.DataFrame, merges: List[Dict[str, Any]], *,
                  how_default: str = "left", copy_base: bool = False) -> pd.DataFrame:
    """
    Merge multiple DataFrames into a base DataFrame in sequence.
    
    Parameters
    ----------
    base : DataFrame
        Starting DataFrame
    merges : list of dict
        Each dict specifies a merge with keys:
        - df (required): DataFrame to merge
        - how: merge strategy (default: how_default)
        - on: column name(s) for same-key merge
        - OR left_on + right_on: different join keys
        - suffixes: tuple for overlapping columns (default: ("", "_y"))
        - drop: columns to drop after merge
    how_default : str
        Default merge strategy (default: "left")
    copy_base : bool
        If True, copy base before modifying
    
    Returns
    -------
    DataFrame
        Merged result
    
    Example
    -------
    >>> out = merge_many_df(
    ...     base=sales_df,
    ...     merges=[
    ...         {"df": buyers_df, "on": "buyer_id"},
    ...         {"df": products_df, "on": "sku_id"},
    ...         {"df": orders_df, "left_on": "order_id", "right_on": "id",
    ...          "suffixes": ("", "_orders"), "drop": ["id"]},
    ...     ],
    ...     how_default="left"
    ... )
    """
    if not isinstance(base, pd.DataFrame):
        raise TypeError("base must be a pandas DataFrame.")
    if not isinstance(merges, list):
        raise TypeError("merges must be a list of dict merge specs.")

    out = base.copy() if copy_base else base

    for idx, spec in enumerate(merges):
        if not isinstance(spec, dict):
            raise TypeError(f"merges[{idx}] must be a dict.")
        if "df" not in spec:
            raise KeyError(f"merges[{idx}] is missing required key: 'df'")

        right = spec["df"]
        if not isinstance(right, pd.DataFrame):
            raise TypeError(f"merges[{idx}]['df'] must be a pandas DataFrame.")

        how = spec.get("how", how_default)
        suffixes = spec.get("suffixes", ("", "_y"))
        drop_cols = spec.get("drop", [])

        if "on" in spec:
            out = out.merge(right, how=how, on=spec["on"], suffixes=suffixes)
        else:
            if "left_on" not in spec or "right_on" not in spec:
                raise KeyError(f"merges[{idx}] must have either 'on' OR both 'left_on' and 'right_on'.")
            out = out.merge(
                right,
                how=how,
                left_on=spec["left_on"],
                right_on=spec["right_on"],
                suffixes=suffixes,
            )

        if drop_cols:
            out = out.drop(columns=[c for c in drop_cols if c in out.columns])

    return out

In [34]:
def revenue_analysis(df: pd.DataFrame, cols):
    """
    Analyze how missing values affect total revenue.
    
    Helps understand if missing data is a significant problem.
    For example, if 5% of rows are missing timezone but represent only
    1% of revenue, it's less concerning.
    
    Parameters
    ----------
    df : DataFrame
        Dataset with order data
    cols : str or list
        Column(s) to analyze for missing values
    
    Notes
    -----
    Uses 'order_row_value' if available, otherwise calculates price * quantity.
    """
    cols = [cols] if isinstance(cols, str) else list(cols)

    row_value = df["order_row_value"] if "order_row_value" in df.columns else (df["price"] * df["quantity"])
    total_value = row_value.sum()

    for col in cols:
        missing_mask = df[col].isna()
        missing_rows = int(missing_mask.sum())
        missing_value = row_value.loc[missing_mask].sum()
        impact_pct = (missing_value / total_value) * 100 if total_value else 0

        print(f"--- Revenue Impact Analysis: {col} ---")
        print(f"Missing Rows:    {missing_rows}")
        print(f"Skipped Value:   ${missing_value:,.2f}")
        print(f"Impact on Total: {impact_pct:.2f}%")
        print("-" * 35)

---
## 10. Timezone Conversion

The company is in Missouri (Central Time), but customers are in different timezones.
These functions convert customer order times to company time.

In [36]:
def to_company_time(order_dt: pd.Series, tz_col: pd.Series,
                    company_tz: str = "America/Chicago") -> pd.Series:
    """
    Convert customer order times to company timezone.
    
    Process:
    1. Group orders by customer timezone
    2. Interpret timestamps as local time in that timezone
    3. Convert to company timezone (America/Chicago)
    4. Remove timezone info (keeping company-local time)
    
    Handles edge cases:
    - Daylight saving time transitions
    - Invalid or missing timezones (falls back to company_tz)
    - Missing order times
    
    Parameters
    ----------
    order_dt : Series
        Order datetime column (naive, no timezone)
    tz_col : Series
        Customer timezone column (e.g., 'America/New_York')
    company_tz : str
        Company timezone (default: 'America/Chicago')
    
    Returns
    -------
    Series
        Times converted to company timezone
    
    Example
    -------
    >>> # Customer in New York orders at 3 PM local time
    >>> # Company time will be 2 PM (Eastern is 1 hour ahead of Central)
    >>> df['order_time_company'] = to_company_time(df['order_datetime'], df['timezone'])
    """
    out = pd.Series(pd.NaT, index=order_dt.index, dtype="datetime64[ns]")

    for tz, idx in tz_col.groupby(tz_col, observed=True).groups.items():
        try:
            z = ZoneInfo(str(tz))
        except Exception:
            z = ZoneInfo(company_tz)

        s = order_dt.loc[idx]
        m = s.notna()
        if m.any():
            s_local = s.loc[m].dt.tz_localize(
                z,
                ambiguous="NaT",
                nonexistent="shift_forward"
            )
            s_company = s_local.dt.tz_convert(ZoneInfo(company_tz)).dt.tz_localize(None)
            out.loc[s.loc[m].index] = s_company

    return out

---
## 11. Missing Value Filling

In [38]:
def fill_categorical_na(df: pd.DataFrame, fill_value: str = "Unknown") -> pd.DataFrame:
    """
    Fill missing values in categorical columns with a placeholder.
    
    Ensures the fill_value is added as a category before filling.
    Modifies df in-place and returns it for chaining.
    
    Parameters
    ----------
    df : DataFrame
        Data to fill
    fill_value : str
        Value to use for missing categories (default: "Unknown")
    
    Returns
    -------
    DataFrame
        Same DataFrame with filled categorical columns
    """
    if not isinstance(df, pd.DataFrame):
        raise TypeError("fill_categorical_na expects a pandas DataFrame.")

    for col in df.columns:
        dtype = df[col].dtype
        if isinstance(dtype, pd.CategoricalDtype):
            if fill_value not in df[col].cat.categories:
                df[col] = df[col].cat.add_categories([fill_value])
            df[col] = df[col].fillna(fill_value)

    return df

In [39]:
def fill_string_na(df: pd.DataFrame, fill_value: str = "Unknown") -> pd.DataFrame:
    """
    Fill missing values in string/object columns (excluding categoricals).
    
    Modifies df in-place and returns it for chaining.
    
    Parameters
    ----------
    df : DataFrame
        Data to fill
    fill_value : str
        Value to use for missing text (default: "Unknown")
    
    Returns
    -------
    DataFrame
        Same DataFrame with filled string columns
    """
    if not isinstance(df, pd.DataFrame):
        raise TypeError("fill_string_na expects a pandas DataFrame.")

    for col in df.columns:
        dtype = df[col].dtype

        # Skip categoricals
        if isinstance(dtype, pd.CategoricalDtype):
            continue

        # Fill string/object columns
        if pd.api.types.is_string_dtype(dtype):
            df[col] = df[col].fillna(fill_value)

    return df

---

## Functions Library Complete

All **22 functions** are loaded and ready to use.

### Quick Usage Examples

```python
# Load data
df = read_csv_dedup('sales.csv')

# Profile it
profile_dashboard(df, "Sales Data")

# Clean and cast types
df = drop_rows_missing_pk(df, 'order_id')
df = typecast_df(df, metadata_cols)

# Visualize
plot_histogram(df['price'], title='Price Distribution')
plot_frequency(df['category'], top_n=10)

# Handle outliers
analyze_numeric_outliers(df['quantity'])
df = trim_outliers_by_percentile(df, 'quantity', p=0.999)

# Merge datasets
result = merge_many_df(base_df, [
    {"df": products_df, "on": "sku_id"},
    {"df": buyers_df, "on": "buyer_id"}
])
```

### Import in Analysis Notebook
```python
%run functions.ipynb
```
